# 02 — CNN14 Training Walkthrough

PANNs-style CNN14 branch trained **from scratch** on kitpri_v2 log-mel spectrograms.

## Architecture
- Input `bn0` (BatchNorm2d on log-mel)
- 6 ConvBlock double-conv stages: 1→64→128→256→512→1024→2048 channels
- Blocks 1–5 average-pool 2×2; block 6 keeps spatial dims
- `AdaptiveAvgPool2d(1)` → flatten → `fc1(2048, 2048)` → `Dropout(0.5)` → `classifier(2048, 1)`
- ~22 M parameters

## Notes from Phase 7 Kaggle notebook
- PANNs weights: only 3/84 layers matched (essentially trains from scratch)
- Individual CNN14 Val F1 ≈ 0.80 before ensemble
- Use `python -m src.train` for the reproducible CLI workflow

In [ ]:
import sys
sys.path.append('..')

import torch
from src.models import CNN14

# Build CNN14 (pretrained=False skips PANNs soft-load)
model = CNN14(dropout=0.5, pretrained=False)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f'Total parameters  : {total_params:.1f} M')
print(f'Trainable         : {trainable:.1f} M')
print(f'Architecture:\n{model}')

In [ ]:
# Forward-pass smoke test with random log-mel: [batch=2, 1-channel, 64-mels, 1000-frames]
dummy_mel = torch.randn(2, 1, 64, 1000)
logits = model(dummy_mel)
print(f'Input shape  : {tuple(dummy_mel.shape)}')
print(f'Output shape : {tuple(logits.shape)}')   # [2]
print(f'Logits       : {logits.detach()}')

In [ ]:
# Freeze backbone — only fc1 + classifier are trainable (Stage 1 behavior)
model.freeze_backbone()
trainable_frozen = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f'Trainable after freeze_backbone(): {trainable_frozen:.1f} M')

# Unfreeze all
model.unfreeze_backbone()
trainable_full = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f'Trainable after unfreeze_backbone(): {trainable_full:.1f} M')

In [ ]:
# Training CLI reference:
# python -m src.train \
#   --config configs/config.yaml \
#   --train-csv data/raw/kitpri-v2/metadata/train.csv \
#   --val-csv   data/raw/kitpri-v2/metadata/val.csv \
#   --data-root data/raw/kitpri-v2
print('See scripts/run_training.sh for the full CLI command.')